In [1]:
from pathlib import Path
import os
import sys

import mne
from pytep import apply_sound, apply_sspsir

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [ ]:
from modules.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from modules.events import get_events_tms_per_task
from modules.preprocessing import fix_stim_artifact_cubic

In [ ]:
raw = mne.io.read_raw_bdf(r"data/raw/V1.bdf", preload=True)

In [ ]:
raw_data = raw.copy()

In [ ]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

In [ ]:
events, event_id = mne.events_from_annotations(raw_data)
event_id, renamed_dict =  convert_dict_trigger(event_id, decode_8bit_trigger)
raw_data.annotations.rename(renamed_dict)
raw_data.event_id = event_id

In [ ]:
raw_data.plot(scalings={"eeg": 100e-6, "emg": 500e-6, "eog": 200e-6}, n_channels=10)

In [ ]:
bad_ch =[]
raw_data.drop_channels(bad_ch)

In [ ]:
events_tms, events_id_tms = get_events_tms_per_task(events, event_id)

In [ ]:
raw_data = fix_stim_artifact_cubic(
    inst=raw_data, 
    events=events,
    event_id=7,
    tmin=-0.005,
    tmax=0.010,
    pre_window=0.010
    post_window=0.010
)

In [ ]:
raw_data.notch_filter(freqs=[60, 120], fir_design='firwin')
raw_data.filter(l_freq=1, h_freq=None, fir_design='firwin')

In [ ]:
epochs = mne.Epochs(
    raw_data,
    events=events_tms,
    event_id=events_id_tms,
    tmin=-0.8,
    tmax=0.8,
    baseline=(-0.25,-0.1),
    preload=True,
    detrend=1,
)

In [ ]:
epochs.average().plot()

In [ ]:
epochs.plot(block=True)

In [ ]:
indices_removidos = [i for i, log in enumerate(epochs.drop_log) if log]

print(f"Foram removidas {len(indices_removidos)} épocas.")
print("Índices:", indices_removidos)

In [ ]:
bad_epochs = []
epochs.drop(bad_epochs)

In [ ]:
from scipy import signal

epochs = epochs.apply_function(signal.detrend, type='linear', picks='all')

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800)
ica.fit(epochs)

In [ ]:
ica.plot_sources(epochs, show_scrollbars=True)

In [ ]:
ica.exclude = [0,1]
epochs_clean = ica.apply(epochs.copy())

Applying ICA to Epochs instance
    Transforming to ICA space (20 components)
    Zeroing out 2 ICA components
    Projecting back using 40 PCA components


C:\Users\marci\AppData\Local\Temp\ipykernel_43936\2884354493.py:2: RuntimeWarning: The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.
  epochs_clean_left = ica.apply(epochs_clean_left.copy())


In [ ]:
epochs_clean.apply_baseline(baseline=(-0.200, -0.05))

In [ ]:
import matlab.engine
eng = matlab.engine.start_matlab()

In [ ]:
epochs_clean = apply_sound(epochs_clean, eng=eng)

In [ ]:
epochs_clean.set_eeg_reference("average", projection=False)

In [ ]:
epochs_clean = apply_sspsir(epochs_clean, eng=eng)
eng.quit()

In [ ]:
epochs_clean = mne.preprocessing.fix_stim_artifact(
    inst=epochs_clean, 
    tmin=-0.005,
    tmax=0.010,
    baseline=(-0.050, -0.01),
    mode='constant'
)

In [ ]:
epochs_clean.filter(l_freq=None, h_freq=80, fir_design='firwin')

In [ ]:
epochs_clean['tms_pulse_rest_bilateral'].copy().crop(tmin=-0.05, tmax=0.300).average().plot(picks=['C3'])

In [ ]:
epochs_clean.save("data/processed/data.fif", overwrite=True)